In [ ]:
# Load necessary libraries
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from sklearn.model_selection import train_test_split

from src.preprocessing import build_preprocessor
from src.models import build_xgboost_model
from src.evaluation import evaluate_model

In [ ]:
# Load the data
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Preprocess the data
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df = df.drop(columns=['customerID'])

In [ ]:
# Split the data
X = df.drop(columns='Churn')
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Build the model
preprocessor = build_preprocessor(X)
model = build_xgboost_model(preprocessor)


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

In [ ]:
param_dist = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 4, 5, 6],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__subsample': [0.7, 0.8, 1.0],
    'classifier__colsample_bytree': [0.7, 0.8, 1.0],
    'classifier__scale_pos_weight': [2.0, 2.5, 3.0, 3.5]
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='recall',
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

In [ ]:
search.fit(X_train, y_train)

In [ ]:
print(search.best_params_)

In [ ]:
best_model = search.best_estimator_

In [ ]:
y_proba = best_model.predict_proba(X_test)[:, 1]

threshold = 0.4
y_pred = (y_proba >= threshold).astype(int)

In [ ]:
evaluate_model(y_test, y_pred, y_proba)